# **Integración y consolidación de nuevas variables**

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## **1. Objetivo**

Una vez construidas las distintas variables derivadas mediante procesos de *feature engineering*, se realizó una etapa final de integración para consolidar todas las *features* en un único dataset maestro.

El objetivo de esta fase fue disponer de una tabla final lista para entrenamiento y validación de modelos predictivos, incorporando información de:
- Tipo de establecimiento
- Complejidad del pedido
- Variables meteorológicas
- Variables geoespaciales y distritos
- Indicador de partición entrenamiento/test

## **2. Datasets utilizados**

Se utilizaron cuatro datasets previamente generados:

- `df_store_type`:	Variables base + tipo de establecimiento

In [ ]:
store_type = '/content/drive/MyDrive/Reto_IA/Store_type/dataset_store_type.csv'
df_store_type = pd.read_csv(store_type)
df_store_type.head()

,store_name,vertical,gtv,delivery_fee,activation_time_local,courier_started_order_local,pickup_time_local,delivery_time_local,tranport_type,saturation,distance_km,hour,day_of_week,is_weekend,day_name,target,target_min,target_class,store_type
0,Pharmacie,WALL - Partner,19.97,4.9,2019-10-01 09:21:34,2019-10-01 09:19:06.342,2019-10-01 09:34:55.904,2019-10-01 09:50:02,MOTORBIKE,66,0.882526,9,1,0,Martes,0 days 00:30:55.658000,30.927633,2,pharmacy
1,Starbucks,WALL - NonPartner,10.50,4.9,2019-10-01 09:18:30,2019-10-01 09:28:04.560,2019-10-01 09:36:02.133,2019-10-01 09:47:39,BICYCLE,40,0.995245,9,1,0,Martes,0 days 00:19:34.440000,19.574000,0,coffee_bakery
2,Starbucks,WALL - NonPartner,10.35,4.9,2019-10-01 15:04:50,2019-10-01 15:04:28.032,2019-10-01 15:25:37.131,2019-10-01 15:34:52,MOTORBIKE,29,1.284852,15,1,0,Martes,0 days 00:30:23.968000,30.399467,2,coffee_bakery
3,NO_STORE,COURIER,6.60,6.6,2019-10-02 10:08:38,2019-10-02 10:11:26.078,2019-10-02 10:30:24.559,2019-10-02 10:50:13,MOTORBIKE,54,3.325588,10,2,0,Miércoles,0 days 00:38:46.922000,38.782033,2,no_store
4,Franprix,WALL - Partner,13.10,3.9,2019-10-02 12:12:09,2019-10-02 12:10:54.226,2019-10-02 12:31:21.010,2019-10-02 12:38:04,MOTORBIKE,77,0.885714,12,2,0,Miércoles,0 days 00:27:09.774000,27.162900,1,supermarket


In [ ]:
df_store_type.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 63589 entries, 0 to 63588
Data columns (total 19 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   store_name                   63589 non-null  object 
 1   vertical                     63589 non-null  object 
 2   gtv                          63589 non-null  float64
 3   delivery_fee                 63589 non-null  float64
 4   activation_time_local        63589 non-null  object 
 5   courier_started_order_local  63589 non-null  object 
 6   pickup_time_local            63589 non-null  object 
 7   delivery_time_local          63589 non-null  object 
 8   tranport_type                63589 non-null  object 
 9   saturation                   63589 non-null  int64  
 10  distance_km                  63589 non-null  float64
 11  hour                         63589 non-null  int64  
 12  day_of_week                  63589 non-null  int64  
 13  is_weekend      

- `df_num_articulos`:	Clima y complejidad del pedido

In [ ]:
num_articulos = '/content/drive/MyDrive/Reto_IA/Clima y Artículos/clima_num_articulos.csv'
df_num_articulos = pd.read_csv(num_articulos)
df_num_articulos.head()

,order_id,customer_id,courier_id,store_address_id,store_name,vertical,gtv,delivery_fee,description,activation_time_local,...,pickup_latitude,pickup_longitude,delivery_latitude,delivery_longitude,saturation,weather_code,precip_mm,weather_desc,Total_Unidades,Articulos_Distintos
0,32.0,700734,441.0,10887.0,Pharmacie,WALL - Partner,19.97,4.9,1 x DOLIPRANE 1000mg - 8 comprimés\n1 x IMODIU...,2019-10-01 09:21:34.000,...,48.861046,2.275441,48.855068,2.283378,66,51.0,0.1,Llovizna ligera,3,3
1,29.0,1648235,239.0,3705.0,Starbucks,WALL - NonPartner,10.50,4.9,"1 x Caffe Latte - Tall, Espresso Roast Class...",2019-10-01 09:18:30.000,...,48.871578,2.304495,48.867017,2.292787,40,51.0,0.1,Llovizna ligera,2,2
2,292.0,13527080,351.0,1035.0,Starbucks,WALL - NonPartner,10.35,4.9,Venti Latte Vanille Glace avec seulement 2 pom...,2019-10-01 15:04:50.000,...,48.855461,2.360776,48.866713,2.364773,29,53.0,0.5,Llovizna moderada,3,3
3,743.0,275173,391.0,NaN,NaN,COURIER,6.60,6.6,Lunettes,2019-10-02 10:08:38.000,...,48.859665,2.285694,48.884335,2.259989,54,0.0,0.0,Despejado,1,1
4,838.0,1105241,263.0,51106.0,Franprix,WALL - Partner,13.10,3.9,"1 x Emmental rape 29% - Franprix - 3x35g - 1,4...",2019-10-02 12:12:09.000,...,48.863417,2.361240,48.868087,2.371050,77,51.0,0.1,Llovizna ligera,5,4


- `df_distritos`:	Variables geográficas y espaciales

In [ ]:
distritos = '/content/drive/MyDrive/Reto_IA/Distritos/distritos.csv'
df_distritos = pd.read_csv(distritos)
df_distritos.head()

,order_id,customer_id,courier_id,store_address_id,store_name,vertical,gtv,delivery_fee,description,activation_time_local,...,pickup_longitude,delivery_latitude,delivery_longitude,saturation,distrito_pickup,distrito_delivery,dist_p_centro,dist_d_centro,cruza_periferico,flujo
0,32.0,700734,441.0,10887.0,Pharmacie,WALL - Partner,19.97,4.9,1 x DOLIPRANE 1000mg - 8 comprimés\n1 x IMODIU...,2019-10-01 09:21:34.000,...,2.275441,48.855068,2.283378,66,16.0,16.0,5.637138,5.038016,0,Centrípeto
1,29.0,1648235,239.0,3705.0,Starbucks,WALL - NonPartner,10.50,4.9,"1 x Caffe Latte - Tall, Espresso Roast Class...",2019-10-01 09:18:30.000,...,2.304495,48.867017,2.292787,40,8.0,16.0,3.866667,4.497923,0,Centrífugo
2,292.0,13527080,351.0,1035.0,Starbucks,WALL - NonPartner,10.35,4.9,Venti Latte Vanille Glace avec seulement 2 pom...,2019-10-01 15:04:50.000,...,2.360776,48.866713,2.364773,29,4.0,11.0,0.640054,1.452745,0,Centrífugo
3,743.0,275173,391.0,NaN,NaN,COURIER,6.60,6.6,Lunettes,2019-10-02 10:08:38.000,...,2.285694,48.884335,2.259989,54,16.0,0.0,4.877379,7.415955,1,Centrífugo
4,838.0,1105241,263.0,51106.0,Franprix,WALL - Partner,13.10,3.9,"1 x Emmental rape 29% - Franprix - 3x35g - 1,4...",2019-10-02 12:12:09.000,...,2.361240,48.868087,2.371050,77,3.0,11.0,1.005934,1.879541,0,Centrífugo


In [ ]:
df_distritos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 63646 entries, 0 to 63645
Data columns (total 27 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   order_id                     63646 non-null  float64
 1   customer_id                  63646 non-null  int64  
 2   courier_id                   63646 non-null  float64
 3   store_address_id             52018 non-null  float64
 4   store_name                   52018 non-null  object 
 5   vertical                     63646 non-null  object 
 6   gtv                          63646 non-null  float64
 7   delivery_fee                 63646 non-null  float64
 8   description                  63614 non-null  object 
 9   activation_time_local        63646 non-null  object 
 10  courier_started_order_local  63642 non-null  object 
 11  pickup_time_local            63646 non-null  object 
 12  delivery_time_local          63646 non-null  object 
 13  tranport_type   

- `df_is_test`:	Flag de separación train/test

In [ ]:
split = '/content/drive/MyDrive/Reto_IA/dataset_modificado_with_split.csv'
df_is_test = pd.read_csv(split)
df_is_test.head()

,store_name,vertical,gtv,delivery_fee,activation_time_local,courier_started_order_local,pickup_time_local,delivery_time_local,tranport_type,saturation,distance_km,hour,day_of_week,is_weekend,day_name,target,target_min,target_class,is_test
0,Pharmacie,WALL - Partner,19.97,4.9,2019-10-01 09:21:34,2019-10-01 09:19:06.342,2019-10-01 09:34:55.904,2019-10-01 09:50:02,MOTORBIKE,66,0.882526,9,1,0,Martes,0 days 00:30:55.658000,30.927633,2,False
1,Starbucks,WALL - NonPartner,10.50,4.9,2019-10-01 09:18:30,2019-10-01 09:28:04.560,2019-10-01 09:36:02.133,2019-10-01 09:47:39,BICYCLE,40,0.995245,9,1,0,Martes,0 days 00:19:34.440000,19.574000,0,False
2,Starbucks,WALL - NonPartner,10.35,4.9,2019-10-01 15:04:50,2019-10-01 15:04:28.032,2019-10-01 15:25:37.131,2019-10-01 15:34:52,MOTORBIKE,29,1.284852,15,1,0,Martes,0 days 00:30:23.968000,30.399467,2,False
3,NO_STORE,COURIER,6.60,6.6,2019-10-02 10:08:38,2019-10-02 10:11:26.078,2019-10-02 10:30:24.559,2019-10-02 10:50:13,MOTORBIKE,54,3.325588,10,2,0,Miércoles,0 days 00:38:46.922000,38.782033,2,False
4,Franprix,WALL - Partner,13.10,3.9,2019-10-02 12:12:09,2019-10-02 12:10:54.226,2019-10-02 12:31:21.010,2019-10-02 12:38:04,MOTORBIKE,77,0.885714,12,2,0,Miércoles,0 days 00:27:09.774000,27.162900,1,True


## **3. Problema de integración**

La integración no podía realizarse utilizando únicamente un identificador simple (`order_id`), debido a que algunos datasets no conservaban exactamente las mismas estructuras o granularidades.

Por ello se definió una clave compuesta basada en variables operativas estables:

In [ ]:
llaves = ['store_name', 'gtv', 'delivery_fee', 'activation_time_local']

Estas variables permiten identificar de manera robusta cada pedido dentro de los distintos datasets.

## **4. Normalización previa de datos y merge**

Antes de realizar los merges fue necesario normalizar los datasets para evitar inconsistencias que pudieran provocar pérdidas de registros durante la unión.

La función `normalizar()` realiza las siguientes tareas:
- Estandarización de `store_name`: se homogeniza el nombre de establecimientos para evitar problemas.
- Estandarización temporal: se redondea el timestamp al minuto para garantizar coincidencia exacta entre datasets.
- Homogeneización numérica: se normalizan variables monetarias para evitar errores de merge producidos por diferencias de precisión decimal.

In [ ]:
def preparar_y_unir_datasets(df_principal, df_art, df_dist, df_split):

    def normalizar(df_input):
        df = df_input.copy()

        # Solo normalizamos si las columnas existen en el DF actual
        if 'store_name' in df.columns:
            df['store_name'] = df['store_name'].fillna('NO_STORE').replace(['', 'nan', 'NaN'], 'NO_STORE')
            df['store_name'] = df['store_name'].astype(str).str.strip().str.upper()

        if 'activation_time_local' in df.columns:
            df['activation_time_local'] = pd.to_datetime(df['activation_time_local']).dt.floor('min')

        if 'gtv' in df.columns:
            df['gtv'] = df['gtv'].round(2)

        if 'delivery_fee' in df.columns:
            df['delivery_fee'] = df['delivery_fee'].round(2)

        return df

    # Normalizar cada uno
    st_clean = normalizar(df_principal)
    art_clean = normalizar(df_art)
    dist_clean = normalizar(df_dist)
    split_clean = normalizar(df_split)

    llaves = ['store_name', 'gtv', 'delivery_fee', 'activation_time_local']

    # Selección de columnas relevantes:
    # Variables incorporadas desde clima y artículos
    cols_art = llaves + ['Total_Unidades', 'Articulos_Distintos', 'weather_code', 'precip_mm', 'weather_desc']
    # Variable sincorporadas desde distritos
    cols_dist = llaves + ['distrito_pickup', 'distrito_delivery', 'dist_p_centro', 'dist_d_centro', 'cruza_periferico', 'flujo']
    # Variable incorporada desde split train/test
    cols_split = llaves + ['is_test']

    # Quitamos duplicados para evitar explosión de filas en el merge
    art_sub = art_clean[cols_art].drop_duplicates(subset=llaves)
    dist_sub = dist_clean[cols_dist].drop_duplicates(subset=llaves)
    split_sub = split_clean[cols_split].drop_duplicates(subset=llaves)

    # MERGE ENCADENADO
    df_merged = st_clean.merge(art_sub, on=llaves, how='left') \
                        .merge(dist_sub, on=llaves, how='left') \
                        .merge(split_sub, on=llaves, how='left')

    # Limpieza final de nulos en columnas numéricas extraídas
    # Las variables derivadas del análisis textual del pedido se imputaron con cero cuando no existía información disponible
    if 'Total_Unidades' in df_merged.columns:
        df_merged['Total_Unidades'] = df_merged['Total_Unidades'].fillna(0).astype(int)
        df_merged['Articulos_Distintos'] = df_merged['Articulos_Distintos'].fillna(0).astype(int)

    return df_merged

df_final = preparar_y_unir_datasets(df_store_type, df_num_articulos, df_distritos, df_is_test)

## **5. Resultado final**

El dataset consolidado contiene:
- 63,589 registros
- 31 variables

In [ ]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 63589 entries, 0 to 63588
Data columns (total 31 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   store_name                   63589 non-null  object        
 1   vertical                     63589 non-null  object        
 2   gtv                          63589 non-null  float64       
 3   delivery_fee                 63589 non-null  float64       
 4   activation_time_local        63589 non-null  datetime64[ns]
 5   courier_started_order_local  63589 non-null  object        
 6   pickup_time_local            63589 non-null  object        
 7   delivery_time_local          63589 non-null  object        
 8   tranport_type                63589 non-null  object        
 9   saturation                   63589 non-null  int64         
 10  distance_km                  63589 non-null  float64       
 11  hour                         63589 non-nu

Finalmente, el dataset consolidado fue exportado a Google Drive para su utilización en etapas posteriores de modelado:

In [ ]:
output_path = '/content/drive/MyDrive/Reto_IA/dataset_nuevas_features.csv'
df_final.to_csv(output_path, index=False)
print(f"DataFrame 'df_final' successfully exported to {output_path}")

DataFrame 'df_final' successfully exported to /content/drive/MyDrive/Reto_IA/dataset_nuevas_features.csv


Este archivo constituye la versión final del dataset enriquecido utilizado para el entrenamiento y evaluación de modelos predictivos de tiempo de entrega.